### Tools

#### Models can request to call tools that perform tasks such as fetching data from a database,searching the web, or running code. Tools are pairings of:
1. A Schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute

In [1]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots don’t “talk” in the same way humans do—they don’t understand language or generate original sentences. What they do is **vocal mimicry**, a sophisticated form of sound copying that serves several purposes in their natural lives.\n\n### 1. Social communication\n- **Flock bonding:** In the wild, parrots live in noisy, densely populated flocks. They use a wide repertoire of calls to keep track of one another, signal food sources, warn of predators, and maintain social cohesion.\n- **Individual recognition:** Each bird’s call has subtle variations that help others identify who’s speaking. Mimicking sounds they hear around them reinforces their place in the group.\n\n### 2. Brain anatomy that supports mimicry\n- **Song system analog:** Parrots possess a set of brain nuclei (the “song system”) similar to songbirds, but it’s even more elaborate. This network includes the **core and shell nuclei** that control both innate calls and learned vocalizations.\n- **Motor co

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather in Boston?")
print(response)

for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'User asks for weather in Boston. We can call get_weather function.', 'tool_calls': [{'id': 'fc_2e9919dc-13f4-48a2-a10b-519b3863789d', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 126, 'total_tokens': 168, 'completion_time': 0.087746581, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.004679236, 'prompt_tokens_details': None, 'queue_time': 0.369004232, 'total_time': 0.092425817}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_ca5edfaab2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a00f20-0326-7e20-90b6-c21e47711ea0-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_2e9919dc-13f4-48a2-a10b-519b3863789d', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'inp

### Tool Execution Loops

In [6]:
# Step1: Model generates tool calls 
messages = [{"role": "user", "content": "What's the weather in Boston"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step3: Pass results back to model for final response
final_response  = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72F and sunny."

The current weather in Boston is sunny. Enjoy your day!


In [7]:
messages

[{'role': 'user', 'content': "What's the weather in Boston"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks "What\'s the weather in Boston". We need to call get_weather function with location "Boston".', 'tool_calls': [{'id': 'fc_11ed54fe-a373-4d41-9223-351ca39f2d81', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 125, 'total_tokens': 175, 'completion_time': 0.104714875, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.018289644, 'prompt_tokens_details': None, 'queue_time': 0.281634524, 'total_time': 0.123004519}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c56a34c3f3', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a00f2f-3657-7102-8732-393af0a20fb9-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bosto